In [ ]:
from dataclasses import dataclass
from typing import Final, Any
from pathlib import Path
import re

import pandas as pd

In [ ]:
# =========================
# SCORESWAY CONFIG
# =========================

SCORESWAY_CODE: Final[str] = "ft1tiv1inq7v1sk3y9tv12yh5"


@dataclass(frozen=True)
class ScoreswaySeasonCode:
    season: str
    tmcl: str
    callback: str | None = None


@dataclass(frozen=True)
class ScoreswayLeague:
    league_name: str
    country_name: str
    slug: str
    seasons: dict[str, ScoreswaySeasonCode]


SCORESWAY_LEAGUES: dict[str, ScoreswayLeague] = {
    "england-premier-league": ScoreswayLeague(
        league_name="Premier League",
        country_name="England",
        slug="england-premier-league",
        seasons={
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="9n12waklv005j8r32sfjj2eqc",
                callback="W3f1a3c26907fc232e65551a74fc12e2f7e98b01ae",
            ),
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="51r6ph2woavlbbpk8f29nynf8",
                callback="W35a065b2dbc6542d92340e574c2e53ab1c324c862",
            ),
        },
    ),

    "netherlands-eredivisie": ScoreswayLeague(
        league_name="Eredivisie",
        country_name="Netherlands",
        slug="netherlands-eredivisie",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="aouykkl1rt7zo06sg0kbzkbh0",
                callback="W32bd64268dbb2b5501cfc9982fc8b1159d6e1bd0a",
            ),
        },
    ),

    "netherlands-eerste-divisie": ScoreswayLeague(
        league_name="Eerste Divisie",
        country_name="Netherlands",
        slug="netherlands-eerste-divisie",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="25u7u2cp66kftzrrneazqp3x0",
                callback="W3532cdb23e419cf470d4449bff5f676fa49f02df7",
            ),
        },
    ),

    "netherlands-tweede-divisie": ScoreswayLeague(
        league_name="Tweede Divisie",
        country_name="Netherlands",
        slug="netherlands-tweede-divisie",
        seasons={
            "2024-25": ScoreswaySeasonCode(
                season="2024-25",
                tmcl="bo5ymeqyjxx2sc6odlkmu1udw",
                callback="W3a847def5d365e44d6e84ff1b56460e348548ac3a",
            ),
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="bk2rzcvvw5kj6zw2ycdisglqs",
                callback="W3ec726c31ab6dfe34dd33f7f3672340ad0f53fbab",
            ),
        },
    ),

    "serbia-superliga": ScoreswayLeague(
        league_name="SuperLiga",
        country_name="Serbia",
        slug="serbia-superliga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="77y4osmpjr44rl7enktiv9qms",
                callback="W371397dbf3bb487d90c55aa68372ab04baeaf2bb7",
            ),
        },
    ),

    "serbia-prva-liga": ScoreswayLeague(
        league_name="Prva Liga",
        country_name="Serbia",
        slug="serbia-prva-liga",
        seasons={
            "2025-26": ScoreswaySeasonCode(
                season="2025-26",
                tmcl="54jd6j2c3t5uksf8s40khsh04",
                callback="W3057af742a38c6ff3229509aee2338d230c3484ae",
            ),
        },
    ),
}

In [ ]:
def slugify_league_name(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def get_scoresway_team_stats_path(
    *,
    league_slug: str,
    season: str,
    output_dir: str | Path = "data/processed",
) -> Path:
    """
    Builds the expected team stats CSV path for one league-season.
    """

    league = SCORESWAY_LEAGUES[league_slug]

    file_prefix = slugify_league_name(f"{league.slug}_{season}")

    return Path(output_dir) / f"{file_prefix}_team_stats.csv"

def read_scoresway_team_stats_csv(
    *,
    league_slug: str,
    season: str,
    output_dir: str | Path = "data/processed",
    encoding: str = "utf-8",
) -> pd.DataFrame:
    """
    Reads one processed Scoresway team stats CSV.
    Adds league metadata columns for easier analysis.
    """

    league = SCORESWAY_LEAGUES[league_slug]

    file_path = get_scoresway_team_stats_path(
        league_slug=league_slug,
        season=season,
        output_dir=output_dir,
    )

    if not file_path.exists():
        raise FileNotFoundError(f"Team stats file not found: {file_path}")

    df = pd.read_csv(file_path, encoding=encoding)

    df["source_league_slug"] = league.slug
    df["source_league_name"] = league.league_name
    df["source_country_name"] = league.country_name
    df["source_season"] = season
    df["source_file"] = str(file_path)

    return df

def read_all_scoresway_team_stats_csvs(
    *,
    scoresway_leagues: dict[str, ScoreswayLeague] = SCORESWAY_LEAGUES,
    output_dir: str | Path = "data/processed",
    encoding: str = "utf-8",
    continue_on_missing: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Reads all processed Scoresway team stats CSV files from SCORESWAY_LEAGUES.

    Returns one combined DataFrame.
    """

    all_dfs = []
    missing_files = []

    for league_slug, league in scoresway_leagues.items():
        for season in league.seasons.keys():
            file_path = get_scoresway_team_stats_path(
                league_slug=league_slug,
                season=season,
                output_dir=output_dir,
            )

            if not file_path.exists():
                missing_files.append(file_path)

                if verbose:
                    print(f"Missing: {file_path}")

                if not continue_on_missing:
                    raise FileNotFoundError(
                        f"Team stats file not found: {file_path}"
                    )

                continue

            if verbose:
                print(f"Reading: {file_path}")

            df = pd.read_csv(file_path, encoding=encoding)

            df["source_league_slug"] = league.slug
            df["source_league_name"] = league.league_name
            df["source_country_name"] = league.country_name
            df["source_season"] = season
            df["source_file"] = str(file_path)

            all_dfs.append(df)

    if not all_dfs:
        raise ValueError(
            "No team stats CSV files were found. "
            f"Checked directory: {Path(output_dir).resolve()}"
        )

    combined_df = pd.concat(all_dfs, ignore_index=True)

    if verbose:
        print("=" * 80)
        print(f"Loaded {len(all_dfs)} files")
        print(f"Combined rows: {len(combined_df)}")
        print(f"Missing files: {len(missing_files)}")
        print("=" * 80)

    return combined_df



In [ ]:
all_team_stats_df = read_all_scoresway_team_stats_csvs(
    output_dir="data/processed",
    continue_on_missing=True,
    verbose=True,
)

all_team_stats_df.head()

In [ ]:
def safe_pct(numerator, denominator):
    return numerator / denominator.replace(0, pd.NA) * 100


def add_scoresway_team_derived_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    games = df["Games Played"].replace(0, pd.NA)

    # =========================
    # Basic output
    # =========================

    per_game_columns = {
        "Goals": "goals_per_game",
        "Goals Conceded": "goals_conceded_per_game",
        "Total Shots": "shots_per_game",
        "Shots On Target ( inc goals )": "shots_on_target_per_game",
        "Total Shots Conceded": "shots_conceded_per_game",
        "Key Passes (Attempt Assists)": "key_passes_per_game",
        "Shots Created": "shots_created_per_game",
        "Final Third Touches": "final_third_touches_per_game",
        "Total Touches In Opposition Box": "box_touches_per_game",
        "Total Passes": "passes_per_game",
        "Open Play Passes": "open_play_passes_per_game",
        "Recoveries": "recoveries_per_game",
        "Interceptions": "interceptions_per_game",
        "Blocks": "blocks_per_game",
        "Total Clearances": "clearances_per_game",
        "Number of Defensive Actions": "defensive_actions_per_game",
        "Duels": "duels_per_game",
        "Aerial Duels": "aerial_duels_per_game",
        "Ground Duels": "ground_duels_per_game",
        "Corners Taken (incl short corners)": "corners_taken_per_game",
        "Set Pieces Goals": "set_piece_goals_per_game",
    }

    for raw_col, new_col in per_game_columns.items():
        if raw_col in df.columns:
            df[new_col] = df[raw_col] / games

    # =========================
    # Shooting
    # =========================

    if {"Shots On Target ( inc goals )", "Total Shots"}.issubset(df.columns):
        df["shots_on_target_pct"] = safe_pct(
            df["Shots On Target ( inc goals )"],
            df["Total Shots"],
        )

    if {"Goals", "Total Shots"}.issubset(df.columns):
        df["goal_conversion_pct"] = safe_pct(
            df["Goals"],
            df["Total Shots"],
        )

    if {"Goals from Inside Box", "Goals"}.issubset(df.columns):
        df["inside_box_goal_share_pct"] = safe_pct(
            df["Goals from Inside Box"],
            df["Goals"],
        )

    # =========================
    # Passing / possession
    # =========================

    if {
        "Total Successful Passes ( Excl Crosses & Corners )",
        "Total Passes",
    }.issubset(df.columns):
        df["pass_accuracy_pct"] = safe_pct(
            df["Total Successful Passes ( Excl Crosses & Corners )"],
            df["Total Passes"],
        )

    if {
        "Successful Passes Opposition Half",
        "Successful Passes Own Half",
    }.issubset(df.columns):
        successful_passes_split = (
            df["Successful Passes Opposition Half"]
            + df["Successful Passes Own Half"]
        )

        df["successful_passes_opposition_half_share_pct"] = safe_pct(
            df["Successful Passes Opposition Half"],
            successful_passes_split,
        )

    if {
        "Successful Long Passes",
        "Unsuccessful Long Passes",
    }.issubset(df.columns):
        total_long_passes = (
            df["Successful Long Passes"]
            + df["Unsuccessful Long Passes"]
        )

        df["long_passes_per_game"] = total_long_passes / games
        df["long_pass_accuracy_pct"] = safe_pct(
            df["Successful Long Passes"],
            total_long_passes,
        )

    if {
        "Successful Short Passes",
        "Unsuccessful Short Passes",
    }.issubset(df.columns):
        total_short_passes = (
            df["Successful Short Passes"]
            + df["Unsuccessful Short Passes"]
        )

        df["short_passes_per_game"] = total_short_passes / games
        df["short_pass_accuracy_pct"] = safe_pct(
            df["Successful Short Passes"],
            total_short_passes,
        )

    # =========================
    # Crossing / wide play
    # =========================

    if {
        "Successful Crosses open play",
        "Unsuccessful Crosses open play",
    }.issubset(df.columns):
        total_open_play_crosses = (
            df["Successful Crosses open play"]
            + df["Unsuccessful Crosses open play"]
        )

        df["open_play_crosses_per_game"] = total_open_play_crosses / games
        df["open_play_cross_accuracy_pct"] = safe_pct(
            df["Successful Crosses open play"],
            total_open_play_crosses,
        )

    # =========================
    # Dribbling
    # =========================

    if {
        "Successful Dribbles",
        "Unsuccessful Dribbles",
    }.issubset(df.columns):
        total_dribbles = (
            df["Successful Dribbles"]
            + df["Unsuccessful Dribbles"]
        )

        df["dribbles_per_game"] = total_dribbles / games
        df["dribble_success_pct"] = safe_pct(
            df["Successful Dribbles"],
            total_dribbles,
        )

    # =========================
    # Duels
    # =========================

    if {"Duels won", "Duels"}.issubset(df.columns):
        df["duels_won_pct"] = safe_pct(
            df["Duels won"],
            df["Duels"],
        )

    if {"Aerial Duels won", "Aerial Duels"}.issubset(df.columns):
        df["aerial_duels_won_pct"] = safe_pct(
            df["Aerial Duels won"],
            df["Aerial Duels"],
        )

    if {"Ground Duels won", "Ground Duels"}.issubset(df.columns):
        df["ground_duels_won_pct"] = safe_pct(
            df["Ground Duels won"],
            df["Ground Duels"],
        )

    if {"Tackles Won", "Tackles Lost"}.issubset(df.columns):
        total_tackles = df["Tackles Won"] + df["Tackles Lost"]

        df["tackles_attempted_per_game"] = total_tackles / games
        df["tackle_success_pct_derived"] = safe_pct(
            df["Tackles Won"],
            total_tackles,
        )

    # =========================
    # Defensive profile
    # =========================

    if {
        "Total Shots Conceded",
        "Total Shots",
    }.issubset(df.columns):
        df["shot_difference_per_game"] = (
            df["Total Shots"] - df["Total Shots Conceded"]
        ) / games

    if {
        "Goals",
        "Goals Conceded",
    }.issubset(df.columns):
        df["goal_difference_per_game"] = (
            df["Goals"] - df["Goals Conceded"]
        ) / games

    if {"Clean Sheets", "Games Played"}.issubset(df.columns):
        df["clean_sheet_rate_pct"] = safe_pct(
            df["Clean Sheets"],
            df["Games Played"],
        )

    # =========================
    # Game state
    # =========================

    if "Points Gained from Losing Positions" in df.columns:
        df["points_gained_from_losing_per_game"] = (
            df["Points Gained from Losing Positions"] / games
        )

    if "Points Dropped from Winning Positions" in df.columns:
        df["points_dropped_from_winning_per_game"] = (
            df["Points Dropped from Winning Positions"] / games
        )

    return df

In [ ]:
all_team_stats_df = add_scoresway_team_derived_metrics(all_team_stats_df)

In [ ]:
TEAM_STYLE_PROFILE_CONFIG = {
    "possession_control": {
        "label": "Possession Control",
        "higher_is_better": [
            "Possession Percentage",
            "passes_per_game",
            "open_play_passes_per_game",
            "pass_accuracy_pct",
        ],
        "lower_is_better": [],
    },

    "territory_penetration": {
        "label": "Territory & Penetration",
        "higher_is_better": [
            "final_third_touches_per_game",
            "box_touches_per_game",
            "successful_passes_opp_half_per_game",
            "key_passes_per_game",
            "shots_created_per_game",
        ],
        "lower_is_better": [],
    },

    "attacking_volume": {
        "label": "Attacking Volume",
        "higher_is_better": [
            "shots_per_game",
            "shots_on_target_per_game",
            "goals_per_game",
            "goal_conversion_pct",
        ],
        "lower_is_better": [],
    },

    "pressing_ball_winning": {
        "label": "Pressing & Ball Winning",
        "higher_is_better": [
            "defensive_actions_per_game",
            "recoveries_per_game",
            "interceptions_per_game",
        ],
        "lower_is_better": [
            "PPDA",
        ],
    },

    "defensive_resistance": {
        "label": "Defensive Resistance",
        "higher_is_better": [
            "clean_sheet_rate_pct",
            "duels_won_pct",
            "tackle_success_pct_derived",
        ],
        "lower_is_better": [
            "goals_conceded_per_game",
            "shots_conceded_per_game",
        ],
    },

    "directness": {
        "label": "Directness",
        "higher_is_better": [
            "long_passes_per_game",
            "launches_per_game",
            "long_pass_share_pct",
        ],
        "lower_is_better": [
            "Possession Percentage",
        ],
    },

    "width_crossing": {
        "label": "Width & Crossing",
        "higher_is_better": [
            "open_play_crosses_per_game",
            "open_play_cross_accuracy_pct",
            "cross_share_pct",
            "corners_taken_per_game",
        ],
        "lower_is_better": [],
    },

    "physicality_duels": {
        "label": "Physicality / Duels",
        "higher_is_better": [
            "duels_per_game",
            "aerial_duels_per_game",
            "ground_duels_per_game",
            "duels_won_pct",
            "aerial_duels_won_pct",
            "ground_duels_won_pct",
        ],
        "lower_is_better": [],
    },

    "set_piece_threat": {
        "label": "Set-Piece Threat",
        "higher_is_better": [
            "set_piece_goals_per_game",
            "set_piece_attempts_per_game",
            "corners_taken_per_game",
            "Successful Corners into Box",
        ],
        "lower_is_better": [],
    },
}

TEAM_STYLE_SCORE_COLUMNS = [
    "possession_control_score",
    "territory_penetration_score",
    "attacking_volume_score",
    "pressing_ball_winning_score",
    "defensive_resistance_score",
    "directness_score",
    "width_crossing_score",
    "physicality_duels_score",
    "set_piece_threat_score",
    "overall_style_intensity_score",
]


TEAM_STYLE_ID_COLUMNS = [
    "contestant_name",
    "source_country_name",
    "source_league_name",
    "source_season",
]

def add_scoresway_team_style_extra_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    games = df["Games Played"].replace(0, pd.NA)

    extra_per_game_columns = {
        "Successful Launches": "successful_launches_per_game",
        "Unsuccessful Launches": "unsuccessful_launches_per_game",
        "Attempts from Set Pieces": "set_piece_attempts_per_game",
        "Successful Passes Opposition Half": "successful_passes_opp_half_per_game",
        "Successful Passes Own Half": "successful_passes_own_half_per_game",
        "Total Losses Of Possession": "losses_of_possession_per_game",
        "Total Fouls Conceded": "fouls_conceded_per_game",
        "Total Fouls Won": "fouls_won_per_game",
    }

    for raw_col, new_col in extra_per_game_columns.items():
        if raw_col in df.columns:
            df[new_col] = df[raw_col] / games

    if {
        "Successful Launches",
        "Unsuccessful Launches",
    }.issubset(df.columns):
        total_launches = df["Successful Launches"] + df["Unsuccessful Launches"]
        df["launches_per_game"] = total_launches / games

    if {
        "Successful Long Passes",
        "Unsuccessful Long Passes",
        "Total Passes",
    }.issubset(df.columns):
        total_long_passes = (
            df["Successful Long Passes"] + df["Unsuccessful Long Passes"]
        )

        df["long_pass_share_pct"] = (
            total_long_passes / df["Total Passes"].replace(0, pd.NA) * 100
        )

    if {
        "Successful Crosses open play",
        "Unsuccessful Crosses open play",
        "Total Passes",
    }.issubset(df.columns):
        total_crosses = (
            df["Successful Crosses open play"]
            + df["Unsuccessful Crosses open play"]
        )

        df["cross_share_pct"] = (
            total_crosses / df["Total Passes"].replace(0, pd.NA) * 100
        )

    return df

def percentile_score(series: pd.Series) -> pd.Series:
    return series.rank(pct=True) * 100


def build_team_style_profile_df(
    df: pd.DataFrame,
    *,
    group_cols: list[str] | None = None,
    config: dict = TEAM_STYLE_PROFILE_CONFIG,
) -> pd.DataFrame:
    """
    Builds team style scores from derived Scoresway team stats.

    Default behaviour:
    Scores are calculated within each league-season, so teams are compared
    to their own competition environment.
    """

    df = df.copy()

    if group_cols is None:
        group_cols = ["source_league_slug", "source_season"]

    score_cols = []

    for style_key, style_data in config.items():
        style_label = style_data["label"]
        score_col = f"{style_key}_score"
        score_cols.append(score_col)

        metric_scores = []

        for col in style_data["higher_is_better"]:
            if col not in df.columns:
                print(f"Missing column for {style_label}: {col}")
                continue

            metric_score = (
                df.groupby(group_cols)[col]
                .transform(percentile_score)
            )

            metric_scores.append(metric_score)

        for col in style_data["lower_is_better"]:
            if col not in df.columns:
                print(f"Missing column for {style_label}: {col}")
                continue

            metric_score = (
                100
                - df.groupby(group_cols)[col]
                .transform(percentile_score)
            )

            metric_scores.append(metric_score)

        if metric_scores:
            df[score_col] = pd.concat(metric_scores, axis=1).mean(axis=1)
        else:
            df[score_col] = pd.NA

    df["overall_style_intensity_score"] = df[score_cols].mean(axis=1)

    return df

def show_team_style_profile(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    season: str | None = None,
) -> pd.DataFrame:
    view = df.copy()

    view = view[
        view["contestant_name"].str.contains(team_name, case=False, na=False)
    ]

    if league_name:
        view = view[
            view["source_league_name"].str.contains(league_name, case=False, na=False)
        ]

    if season:
        view = view[view["source_season"] == season]

    cols = TEAM_STYLE_ID_COLUMNS + TEAM_STYLE_SCORE_COLUMNS

    return view[cols].sort_values(
        "overall_style_intensity_score",
        ascending=False,
    )


In [ ]:
all_team_stats_df = add_scoresway_team_style_extra_metrics(all_team_stats_df)

In [ ]:
# team_style_df = build_team_style_profile_df(all_team_stats_df)
team_style_df = build_team_style_profile_df(
    all_team_stats_df,
    group_cols=["source_league_slug", "source_season"],
)

team_style_df.sample(5)

In [ ]:
# team_style_df.columns.to_list()
# team_style_df["season_name"].unique()
eredivisie_df = team_style_df[(team_style_df["competition_name"] == "Eredivisie") & (team_style_df["season_name"] == "2025/2026")].copy()

team_style_overview_df = team_style_df[
    TEAM_STYLE_ID_COLUMNS + TEAM_STYLE_SCORE_COLUMNS
].copy()

# team_style_overview_df.sort_values(
#     "overall_style_intensity_score",
#     ascending=False,
# ).head(20)

In [ ]:
# Best possession teams
eredivisie_df[
    [
        "contestant_name",
        "source_league_name",
        "source_season",
        "possession_control_score",
        "Possession Percentage",
        "passes_per_game",
        "pass_accuracy_pct",
    ]
].sort_values("possession_control_score", ascending=False).head(10)

# # Most direct teams
# eredivisie_df[
#     [
#         "contestant_name",
#         "source_league_name",
#         "source_season",
#         "directness_score",
#         "Possession Percentage",
#         "long_passes_per_game",
#         "launches_per_game",
#         "long_pass_share_pct",
#     ]
# ].sort_values("directness_score", ascending=False).head(20)

# # Highest pressing teams
# eredivisie_df[
#     [
#         "contestant_name",
#         "source_league_name",
#         "source_season",
#         "pressing_ball_winning_score",
#         "PPDA",
#         "defensive_actions_per_game",
#         "recoveries_per_game",
#         "interceptions_per_game",
#     ]
# ].sort_values("pressing_ball_winning_score", ascending=False).head(20)

# # Best defensive resistance
# eredivisie_df[
#     [
#         "contestant_name",
#         "source_league_name",
#         "source_season",
#         "defensive_resistance_score",
#         "goals_conceded_per_game",
#         "shots_conceded_per_game",
#         "clean_sheet_rate_pct",
#     ]
# ].sort_values("defensive_resistance_score", ascending=False).head(20)

In [ ]:
show_team_style_profile(
    eredivisie_df,
    team_name="Ajax",
    season="2025-26",
)

In [ ]:
# =========================
# LEAGUE COMPARISON CONFIG
# =========================

LEAGUE_COMPARISON_METRICS = {
    # Possession / passing
    "Possession Percentage": "avg_possession_pct",
    "passes_per_game": "avg_passes_per_game",
    "open_play_passes_per_game": "avg_open_play_passes_per_game",
    "pass_accuracy_pct": "avg_pass_accuracy_pct",
    "successful_passes_opposition_half_share_pct": "avg_opp_half_pass_share_pct",

    # Territory / attack
    "final_third_touches_per_game": "avg_final_third_touches_per_game",
    "box_touches_per_game": "avg_box_touches_per_game",
    "shots_per_game": "avg_shots_per_game",
    "shots_on_target_per_game": "avg_shots_on_target_per_game",
    "goals_per_game": "avg_goals_per_game",
    "key_passes_per_game": "avg_key_passes_per_game",
    "shots_created_per_game": "avg_shots_created_per_game",

    # Pressing / defensive activity
    "PPDA": "avg_ppda",
    "defensive_actions_per_game": "avg_defensive_actions_per_game",
    "recoveries_per_game": "avg_recoveries_per_game",
    "interceptions_per_game": "avg_interceptions_per_game",
    "tackles_attempted_per_game": "avg_tackles_attempted_per_game",

    # Defensive pressure faced
    "shots_conceded_per_game": "avg_shots_conceded_per_game",
    "goals_conceded_per_game": "avg_goals_conceded_per_game",
    "clearances_per_game": "avg_clearances_per_game",
    "blocks_per_game": "avg_blocks_per_game",

    # Directness
    "long_passes_per_game": "avg_long_passes_per_game",
    "launches_per_game": "avg_launches_per_game",
    "long_pass_share_pct": "avg_long_pass_share_pct",

    # Width / crossing
    "open_play_crosses_per_game": "avg_open_play_crosses_per_game",
    "open_play_cross_accuracy_pct": "avg_open_play_cross_accuracy_pct",
    "cross_share_pct": "avg_cross_share_pct",
    "corners_taken_per_game": "avg_corners_taken_per_game",

    # Physicality / duels
    "duels_per_game": "avg_duels_per_game",
    "ground_duels_per_game": "avg_ground_duels_per_game",
    "aerial_duels_per_game": "avg_aerial_duels_per_game",
    "duels_won_pct": "avg_duels_won_pct",
    "aerial_duels_won_pct": "avg_aerial_duels_won_pct",
    "ground_duels_won_pct": "avg_ground_duels_won_pct",
    "fouls_conceded_per_game": "avg_fouls_conceded_per_game",

    # Set pieces
    "set_piece_goals_per_game": "avg_set_piece_goals_per_game",
    "set_piece_attempts_per_game": "avg_set_piece_attempts_per_game",
}

# =========================
# LEAGUE ENVIRONMENT CONFIG
# =========================

LEAGUE_ENVIRONMENT_SCORE_CONFIG = {
    "possession_environment": {
        "label": "Possession Environment",
        "higher_is_better": [
            "avg_possession_pct",
            "avg_passes_per_game",
            "avg_open_play_passes_per_game",
            "avg_pass_accuracy_pct",
        ],
        "lower_is_better": [],
    },

    "attacking_volume_environment": {
        "label": "Attacking Volume Environment",
        "higher_is_better": [
            "avg_shots_per_game",
            "avg_shots_on_target_per_game",
            "avg_goals_per_game",
            "avg_key_passes_per_game",
            "avg_shots_created_per_game",
        ],
        "lower_is_better": [],
    },

    "territory_environment": {
        "label": "Territory Environment",
        "higher_is_better": [
            "avg_final_third_touches_per_game",
            "avg_box_touches_per_game",
            "avg_opp_half_pass_share_pct",
        ],
        "lower_is_better": [],
    },

    "pressing_environment": {
        "label": "Pressing Environment",
        "higher_is_better": [
            "avg_defensive_actions_per_game",
            "avg_recoveries_per_game",
            "avg_interceptions_per_game",
        ],
        "lower_is_better": [
            "avg_ppda",
        ],
    },

    "directness_environment": {
        "label": "Directness Environment",
        "higher_is_better": [
            "avg_long_passes_per_game",
            "avg_launches_per_game",
            "avg_long_pass_share_pct",
        ],
        "lower_is_better": [
            "avg_possession_pct",
        ],
    },

    "wide_play_environment": {
        "label": "Wide Play Environment",
        "higher_is_better": [
            "avg_open_play_crosses_per_game",
            "avg_cross_share_pct",
            "avg_corners_taken_per_game",
        ],
        "lower_is_better": [],
    },

    "physicality_environment": {
        "label": "Physicality Environment",
        "higher_is_better": [
            "avg_duels_per_game",
            "avg_ground_duels_per_game",
            "avg_aerial_duels_per_game",
            "avg_fouls_conceded_per_game",
        ],
        "lower_is_better": [],
    },

    "set_piece_environment": {
        "label": "Set-Piece Environment",
        "higher_is_better": [
            "avg_set_piece_goals_per_game",
            "avg_set_piece_attempts_per_game",
            "avg_corners_taken_per_game",
        ],
        "lower_is_better": [],
    },
}

# =========================
# LEAGUE OVERVIEW COLUMNS
# =========================

LEAGUE_ID_COLUMNS = [
    "source_country_name",
    "source_league_name",
    "source_season",
    "teams_count",
]

LEAGUE_ENVIRONMENT_SCORE_COLUMNS = [
    "possession_environment_score",
    "attacking_volume_environment_score",
    "territory_environment_score",
    "pressing_environment_score",
    "directness_environment_score",
    "wide_play_environment_score",
    "physicality_environment_score",
    "set_piece_environment_score",
    "overall_league_environment_score",
]

LEAGUE_STYLE_OVERVIEW_COLUMNS = (
    LEAGUE_ID_COLUMNS + LEAGUE_ENVIRONMENT_SCORE_COLUMNS
)

# =========================
# BUILD LEAGUE COMPARISON DF
# =========================

def build_league_comparison_df(
    df: pd.DataFrame,
    *,
    group_cols: list[str] | None = None,
    metrics_config: dict[str, str] = LEAGUE_COMPARISON_METRICS,
) -> pd.DataFrame:
    """
    Builds one row per league-season using average team metrics.

    This creates the base table for comparing league environments.
    """

    df = df.copy()

    if group_cols is None:
        group_cols = [
            "source_country_name",
            "source_league_name",
            "source_league_slug",
            "source_season",
        ]

    # Count teams
    team_id_col = "contestant_id" if "contestant_id" in df.columns else "contestant_name"

    agg_dict = {
        "teams_count": (team_id_col, "nunique"),
    }

    # Average only columns that exist
    for source_col, output_col in metrics_config.items():
        if source_col in df.columns:
            agg_dict[output_col] = (source_col, "mean")
        else:
            print(f"Missing league comparison column: {source_col}")

    league_df = (
        df.groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )

    return league_df

# =========================
# ADD LEAGUE ENVIRONMENT SCORES
# =========================

def add_league_environment_scores(
    df: pd.DataFrame,
    *,
    config: dict = LEAGUE_ENVIRONMENT_SCORE_CONFIG,
) -> pd.DataFrame:
    """
    Adds percentile-based league environment scores.

    These scores compare leagues against each other.
    Higher score means the league has more of that characteristic.
    """

    df = df.copy()

    score_cols = []

    for score_key, score_data in config.items():
        score_col = f"{score_key}_score"
        score_cols.append(score_col)

        metric_scores = []

        # Higher value means stronger trait
        for col in score_data["higher_is_better"]:
            if col not in df.columns:
                print(f"Missing column for {score_data['label']}: {col}")
                continue

            metric_scores.append(df[col].rank(pct=True) * 100)

        # Lower value means stronger trait
        for col in score_data["lower_is_better"]:
            if col not in df.columns:
                print(f"Missing column for {score_data['label']}: {col}")
                continue

            metric_scores.append(100 - (df[col].rank(pct=True) * 100))

        if metric_scores:
            df[score_col] = pd.concat(metric_scores, axis=1).mean(axis=1)
        else:
            df[score_col] = pd.NA

    df["overall_league_environment_score"] = df[score_cols].mean(axis=1)

    return df

# =========================
# SHOW LEAGUE PROFILE
# =========================

def show_league_profile(
    df: pd.DataFrame,
    *,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    league_slug: str | None = None,
) -> dict[str, pd.DataFrame]:
    """
    Shows a compact league environment profile.

    Returns:
    - overview scores
    - raw possession/attack indicators
    - raw pressing/defensive indicators
    - raw directness/physicality indicators
    """

    view = df.copy()

    # Filter by league slug
    if league_slug:
        view = view[
            view["source_league_slug"].str.contains(
                league_slug,
                case=False,
                na=False,
            )
        ]

    # Filter by league name
    if league_name:
        view = view[
            view["source_league_name"].str.contains(
                league_name,
                case=False,
                na=False,
            )
        ]

    # Filter by country
    if country_name:
        view = view[
            view["source_country_name"].str.contains(
                country_name,
                case=False,
                na=False,
            )
        ]

    # Filter by season
    if season:
        view = view[view["source_season"] == season]

    if view.empty:
        raise ValueError("No league found with the selected filters.")

    id_cols = [
        "source_country_name",
        "source_league_name",
        "source_season",
        "teams_count",
    ]

    score_cols = [
        "possession_environment_score",
        "attacking_volume_environment_score",
        "territory_environment_score",
        "pressing_environment_score",
        "directness_environment_score",
        "wide_play_environment_score",
        "physicality_environment_score",
        "set_piece_environment_score",
        "overall_league_environment_score",
    ]

    possession_attack_cols = [
        "avg_possession_pct",
        "avg_passes_per_game",
        "avg_pass_accuracy_pct",
        "avg_final_third_touches_per_game",
        "avg_shots_per_game",
        "avg_shots_on_target_per_game",
        "avg_goals_per_game",
        "avg_key_passes_per_game",
    ]

    pressing_defence_cols = [
        "avg_ppda",
        "avg_defensive_actions_per_game",
        "avg_recoveries_per_game",
        "avg_interceptions_per_game",
        "avg_shots_conceded_per_game",
        "avg_goals_conceded_per_game",
        "avg_clearances_per_game",
        "avg_blocks_per_game",
    ]

    direct_physical_cols = [
        "avg_long_passes_per_game",
        "avg_launches_per_game",
        "avg_long_pass_share_pct",
        "avg_open_play_crosses_per_game",
        "avg_cross_share_pct",
        "avg_duels_per_game",
        "avg_aerial_duels_per_game",
        "avg_ground_duels_per_game",
        "avg_fouls_conceded_per_game",
    ]

    # Keep only available columns
    def available(cols: list[str]) -> list[str]:
        return [col for col in cols if col in view.columns]

    overview = view[available(id_cols + score_cols)].copy()
    possession_attack = view[available(id_cols + possession_attack_cols)].copy()
    pressing_defence = view[available(id_cols + pressing_defence_cols)].copy()
    direct_physical = view[available(id_cols + direct_physical_cols)].copy()

    return {
        "overview": overview,
        "possession_attack": possession_attack,
        "pressing_defence": pressing_defence,
        "direct_physical": direct_physical,
    }



In [ ]:
# =========================
# CREATE LEAGUE COMPARISON DF
# =========================

league_comparison_df = build_league_comparison_df(all_team_stats_df)

# league_comparison_df.head()

# =========================
# CREATE FINAL LEAGUE DASHBOARD DF
# =========================

league_comparison_df = add_league_environment_scores(league_comparison_df)

# league_comparison_df.head()

# =========================
# LEAGUE STYLE OVERVIEW
# =========================

league_style_overview_df = league_comparison_df[
    LEAGUE_STYLE_OVERVIEW_COLUMNS
].copy()

league_style_overview_df.sort_values(
    "overall_league_environment_score",
    ascending=False,
)

In [ ]:
# =========================
# MOST POSSESSION-BASED LEAGUES
# =========================

# league_comparison_df[
#     [
#         "source_country_name",
#         "source_league_name",
#         "source_season",
#         "possession_environment_score",
#         "avg_possession_pct",
#         "avg_passes_per_game",
#         "avg_pass_accuracy_pct",
#     ]
# ].sort_values("possession_environment_score", ascending=False)

# =========================
# MOST DIRECT LEAGUES
# =========================

league_comparison_df[
    [
        "source_country_name",
        "source_league_name",
        "source_season",
        "directness_environment_score",
        "avg_long_passes_per_game",
        # "avg_launches_per_game",
        # "avg_long_pass_share_pct",
        "avg_possession_pct",
    ]
].sort_values("directness_environment_score", ascending=False)

In [ ]:
# =========================
# VIEW ONE LEAGUE PROFILE
# =========================

league_profile = show_league_profile(
    league_comparison_df,
    league_name="Eredivisie",
    season="2025-26",
)

league_profile["overview"]

In [ ]:
league_profile = show_league_profile(
    league_comparison_df,
    country_name="Serbia",
    league_name="SuperLiga",
    season="2025-26",
)

league_profile["overview"]

In [ ]:
# =========================
# TEAM PROFILE SUMMARY CONFIG
# =========================

TEAM_STYLE_SCORE_LABELS = {
    "possession_control_score": "Possession Control",
    "territory_penetration_score": "Territory & Penetration",
    "attacking_volume_score": "Attacking Volume",
    "pressing_ball_winning_score": "Pressing & Ball Winning",
    "defensive_resistance_score": "Defensive Resistance",
    "directness_score": "Directness",
    "width_crossing_score": "Width & Crossing",
    "physicality_duels_score": "Physicality / Duels",
    "set_piece_threat_score": "Set-Piece Threat",
    "overall_style_intensity_score": "Overall Style Intensity",
}


TEAM_PROFILE_METRIC_GROUPS = {
    "possession_build_up": {
        "label": "Possession / Build-up",
        "columns": [
            "Possession Percentage",
            "passes_per_game",
            "open_play_passes_per_game",
            "pass_accuracy_pct",
            "successful_passes_opposition_half_share_pct",
            "successful_passes_opp_half_per_game",
        ],
    },

    "territory_attack": {
        "label": "Territory / Attack",
        "columns": [
            "goals_per_game",
            "shots_per_game",
            "shots_on_target_per_game",
            "goal_conversion_pct",
            "key_passes_per_game",
            "shots_created_per_game",
            "final_third_touches_per_game",
            "box_touches_per_game",
        ],
    },

    "pressing_defending": {
        "label": "Pressing / Defending",
        "columns": [
            "PPDA",
            "defensive_actions_per_game",
            "recoveries_per_game",
            "interceptions_per_game",
            "shots_conceded_per_game",
            "goals_conceded_per_game",
            "clean_sheet_rate_pct",
            "clearances_per_game",
            "blocks_per_game",
        ],
    },

    "directness_width": {
        "label": "Directness / Width",
        "columns": [
            "long_passes_per_game",
            "launches_per_game",
            "long_pass_share_pct",
            "open_play_crosses_per_game",
            "open_play_cross_accuracy_pct",
            "cross_share_pct",
            "corners_taken_per_game",
        ],
    },

    "duels_physicality": {
        "label": "Duels / Physicality",
        "columns": [
            "duels_per_game",
            "aerial_duels_per_game",
            "ground_duels_per_game",
            "duels_won_pct",
            "aerial_duels_won_pct",
            "ground_duels_won_pct",
            "tackles_attempted_per_game",
            "tackle_success_pct_derived",
        ],
    },
}

# =========================
# SMALL HELPERS
# =========================

def _available_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [col for col in columns if col in df.columns]


def _format_metric_name(value: str) -> str:
    return (
        value.replace("_", " ")
        .replace("pct", "%")
        .replace("ppda", "PPDA")
        .title()
    )


def _filter_team_rows(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
) -> pd.DataFrame:
    view = df.copy()

    if exact_match:
        view = view[
            view["contestant_name"].str.lower() == team_name.lower()
        ]
    else:
        view = view[
            view["contestant_name"].str.contains(team_name, case=False, na=False)
        ]

    if league_name:
        view = view[
            view["source_league_name"].str.contains(league_name, case=False, na=False)
        ]

    if country_name:
        view = view[
            view["source_country_name"].str.contains(country_name, case=False, na=False)
        ]

    if season:
        view = view[view["source_season"] == season]

    return view

# =========================
# TEAM PROFILE SUMMARY FUNCTION
# =========================

def show_team_profile_summary(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
    top_n_traits: int = 3,
) -> dict[str, pd.DataFrame]:
    """
    Returns a clean team profile summary from team_style_df.

    Use this in Jupyter to inspect:
    - overview
    - style scores
    - strongest traits
    - weakest traits
    - key metrics
    - league rankings
    """

    matches = _filter_team_rows(
        df,
        team_name=team_name,
        league_name=league_name,
        country_name=country_name,
        season=season,
        exact_match=exact_match,
    )

    if matches.empty:
        raise ValueError("No team found with the selected filters.")

    candidate_cols = _available_columns(
        matches,
        [
            "contestant_name",
            "source_country_name",
            "source_league_name",
            "source_season",
        ],
    )

    # If there are multiple matches, return candidates instead of guessing
    if len(matches) > 1:
        return {
            "candidates": matches[candidate_cols].drop_duplicates().reset_index(drop=True)
        }

    team_row = matches.iloc[0]

    league_mask = (
        (df["source_league_slug"] == team_row["source_league_slug"])
        & (df["source_season"] == team_row["source_season"])
    )

    league_df = df[league_mask].copy()
    league_teams_count = league_df["contestant_name"].nunique()

    # =========================
    # Overview
    # =========================

    overview_cols = _available_columns(
        df,
        [
            "contestant_name",
            "source_country_name",
            "source_league_name",
            "source_season",
            "Games Played",
            "Goals",
            "Goals Conceded",
            "Clean Sheets",
            "Possession Percentage",
            "PPDA",
        ],
    )

    overview = pd.DataFrame([team_row[overview_cols]])

    # =========================
    # Style scores
    # =========================

    style_rows = []

    for score_col, label in TEAM_STYLE_SCORE_LABELS.items():
        if score_col not in df.columns:
            continue

        value = team_row[score_col]

        rank = (
            league_df[score_col]
            .rank(ascending=False, method="min")
            .loc[team_row.name]
        )

        style_rows.append(
            {
                "style_area": label,
                "score": value,
                "league_rank": int(rank) if pd.notna(rank) else pd.NA,
                "league_teams": league_teams_count,
            }
        )

    style_scores = (
        pd.DataFrame(style_rows)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

    strongest_traits = style_scores.head(top_n_traits).reset_index(drop=True)

    weakest_traits = (
        style_scores
        .sort_values("score", ascending=True)
        .head(top_n_traits)
        .reset_index(drop=True)
    )

    # =========================
    # Key metrics by area
    # =========================

    key_metric_rows = []

    for group_key, group_data in TEAM_PROFILE_METRIC_GROUPS.items():
        group_label = group_data["label"]

        for col in group_data["columns"]:
            if col not in df.columns:
                continue

            value = team_row[col]

            rank = (
                league_df[col]
                .rank(ascending=False, method="min")
                .loc[team_row.name]
            )

            key_metric_rows.append(
                {
                    "area": group_label,
                    "metric": _format_metric_name(col),
                    "column": col,
                    "value": value,
                    "league_rank": int(rank) if pd.notna(rank) else pd.NA,
                    "league_teams": league_teams_count,
                }
            )

    key_metrics = pd.DataFrame(key_metric_rows)

    # =========================
    # League rankings only
    # =========================

    league_rankings = key_metrics[
        [
            "area",
            "metric",
            "value",
            "league_rank",
            "league_teams",
        ]
    ].copy()

    return {
        "overview": overview.reset_index(drop=True),
        "style_scores": style_scores,
        "strongest_traits": strongest_traits,
        "weakest_traits": weakest_traits,
        "key_metrics": key_metrics,
        "league_rankings": league_rankings,
    }

# =========================
# SHOW ONE TEAM PROFILE
# =========================

profile = show_team_profile_summary(
    team_style_df,
    team_name="PSV",
    league_name="Eredivisie",
    season="2025-26",
)

profile["style_scores"]

In [ ]:
# profile["style_scores"]
# profile["strongest_traits"]
# profile["weakest_traits"]
profile["key_metrics"]

In [ ]:
# =========================
# OPPOSITION REPORT CONFIG
# =========================

OPPOSITION_STYLE_INTERPRETATION = {
    "possession_control_score": {
        "high": "Ball-dominant profile. Prepare for longer possession phases and sustained circulation.",
        "medium": "Balanced possession profile. They can keep the ball, but are not necessarily dominant.",
        "low": "Less possession-oriented profile. They may be more comfortable without long spells of control.",
    },

    "territory_penetration_score": {
        "high": "Strong territory profile. They reach advanced areas often and can sustain pressure.",
        "medium": "Moderate territory profile. They can access advanced zones, but not consistently dominant.",
        "low": "Limited territory profile. They may struggle to sustain attacks high up the pitch.",
    },

    "attacking_volume_score": {
        "high": "High attacking volume. They generate frequent shots and chance-creation actions.",
        "medium": "Moderate attacking volume. Their chance creation is present but not overwhelming.",
        "low": "Low attacking volume. They may rely more on efficiency, transitions or set pieces.",
    },

    "pressing_ball_winning_score": {
        "high": "Aggressive ball-winning profile. Build-up security and spacing under pressure are important.",
        "medium": "Moderate pressing profile. They can press, but it may not define their whole game model.",
        "low": "Lower pressing profile. They may defend in more controlled or deeper phases.",
    },

    "defensive_resistance_score": {
        "high": "Strong defensive resistance. They are harder to break down and limit opposition output well.",
        "medium": "Moderate defensive resistance. They are not clearly weak, but can still be attacked.",
        "low": "Defensive vulnerability indicator. There may be space to create shots or sustain pressure.",
    },

    "directness_score": {
        "high": "Direct profile. Prepare for longer passes, second balls and quick territory gains.",
        "medium": "Mixed directness profile. They can go long but are not fully direct.",
        "low": "Less direct profile. They are less reliant on long passes or launch volume.",
    },

    "width_crossing_score": {
        "high": "Wide and crossing-oriented profile. Wide defending, box protection and far-post coverage matter.",
        "medium": "Moderate wide-play profile. They use width, but it may not be their main route.",
        "low": "Lower crossing profile. They may attack more centrally or through other routes.",
    },

    "physicality_duels_score": {
        "high": "Duel-heavy profile. Physical preparation, aerial control and second-ball structure are important.",
        "medium": "Moderate duel profile. Physical contests matter, but do not fully define the game.",
        "low": "Lower duel-volume profile. The match may be less based on repeated physical contests.",
    },

    "set_piece_threat_score": {
        "high": "Set-piece threat. Defensive set-piece preparation should be a priority.",
        "medium": "Moderate set-piece threat. They can be dangerous, but it may not be a core weapon.",
        "low": "Lower set-piece threat. Set pieces still matter, but they are not a standout strength.",
    },
}

# =========================
# OPPOSITION REPORT HELPERS
# =========================

def _score_to_band(score: float) -> str:
    """
    Converts a style score into a simple interpretation band.
    """

    if pd.isna(score):
        return "unknown"

    if score >= 70:
        return "high"

    if score <= 40:
        return "low"

    return "medium"


def _get_style_score_from_profile(
    profile: dict[str, pd.DataFrame],
    *,
    score_col: str,
) -> float | None:
    """
    Reads one style score from the profile summary output.
    """

    style_scores = profile["style_scores"].copy()

    label = TEAM_STYLE_SCORE_LABELS.get(score_col)

    if label is None:
        return None

    row = style_scores[style_scores["style_area"] == label]

    if row.empty:
        return None

    return row.iloc[0]["score"]


def _build_style_interpretation_rows(
    profile: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    """
    Converts style scores into football interpretation rows.
    """

    rows = []

    for score_col, interpretations in OPPOSITION_STYLE_INTERPRETATION.items():
        score = _get_style_score_from_profile(
            profile,
            score_col=score_col,
        )

        if score is None:
            continue

        band = _score_to_band(score)
        label = TEAM_STYLE_SCORE_LABELS.get(score_col, score_col)

        interpretation = interpretations.get(
            band,
            "No interpretation available.",
        )

        rows.append(
            {
                "style_area": label,
                "score": score,
                "band": band,
                "interpretation": interpretation,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )


def _get_top_report_points(
    interpretation_df: pd.DataFrame,
    *,
    min_score: float = 70,
    max_points: int = 4,
) -> list[str]:
    """
    Returns strongest opposition points as short text bullets.
    """

    view = interpretation_df[
        interpretation_df["score"] >= min_score
    ].copy()

    view = view.sort_values("score", ascending=False).head(max_points)

    return [
        f"{row['style_area']}: {row['interpretation']}"
        for _, row in view.iterrows()
    ]


def _get_weakness_report_points(
    interpretation_df: pd.DataFrame,
    *,
    max_score: float = 40,
    max_points: int = 4,
) -> list[str]:
    """
    Returns lower-scoring areas as possible vulnerability indicators.
    """

    view = interpretation_df[
        interpretation_df["score"] <= max_score
    ].copy()

    view = view.sort_values("score", ascending=True).head(max_points)

    return [
        f"{row['style_area']}: {row['interpretation']}"
        for _, row in view.iterrows()
    ]

# =========================
# OPPOSITION REPORT HELPER
# =========================

def build_opposition_report_helper(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
) -> dict[str, Any]:
    """
    Builds a structured opposition report helper for one team.

    This uses:
    - show_team_profile_summary()
    - team style scores
    - key metrics
    - league rankings

    Output:
    - overview
    - style interpretation
    - attacking profile
    - defensive profile
    - main strengths
    - possible vulnerabilities
    - preparation focus
    - key metrics
    """

    profile = show_team_profile_summary(
        df,
        team_name=team_name,
        league_name=league_name,
        country_name=country_name,
        season=season,
        exact_match=exact_match,
    )

    # If more than one team matched, return candidates
    if "candidates" in profile:
        return profile

    interpretation_df = _build_style_interpretation_rows(profile)

    attacking_areas = [
        "Possession Control",
        "Territory & Penetration",
        "Attacking Volume",
        "Directness",
        "Width & Crossing",
        "Set-Piece Threat",
    ]

    defensive_areas = [
        "Pressing & Ball Winning",
        "Defensive Resistance",
        "Physicality / Duels",
    ]

    attacking_profile = interpretation_df[
        interpretation_df["style_area"].isin(attacking_areas)
    ].reset_index(drop=True)

    defensive_profile = interpretation_df[
        interpretation_df["style_area"].isin(defensive_areas)
    ].reset_index(drop=True)

    main_strengths = _get_top_report_points(
        interpretation_df,
        min_score=70,
        max_points=4,
    )

    possible_vulnerabilities = _get_weakness_report_points(
        interpretation_df,
        max_score=40,
        max_points=4,
    )

    # Fallback if no very high or very low scores exist
    if not main_strengths:
        strongest = profile["strongest_traits"].copy()
        main_strengths = [
            f"{row['style_area']}: one of their strongest relative profile areas."
            for _, row in strongest.iterrows()
        ]

    if not possible_vulnerabilities:
        weakest = profile["weakest_traits"].copy()
        possible_vulnerabilities = [
            f"{row['style_area']}: one of their lower relative profile areas."
            for _, row in weakest.iterrows()
        ]

    preparation_focus = []

    high_areas = interpretation_df[interpretation_df["score"] >= 70]["style_area"].tolist()

    if "Possession Control" in high_areas:
        preparation_focus.append(
            "Prepare compact defensive spacing and clear pressing triggers against sustained possession."
        )

    if "Territory & Penetration" in high_areas:
        preparation_focus.append(
            "Protect central access and prevent repeated final-third entries."
        )

    if "Attacking Volume" in high_areas:
        preparation_focus.append(
            "Limit shot volume early by controlling second balls and defending the edge of the box."
        )

    if "Pressing & Ball Winning" in high_areas:
        preparation_focus.append(
            "Build-up structure must be secure, with clear support angles and escape routes."
        )

    if "Directness" in high_areas:
        preparation_focus.append(
            "Prepare for direct balls, depth protection and second-ball reactions."
        )

    if "Width & Crossing" in high_areas:
        preparation_focus.append(
            "Wide defending and box occupation are key, especially far-post protection."
        )

    if "Physicality / Duels" in high_areas:
        preparation_focus.append(
            "Duel preparation matters, especially aerial contests and loose-ball reactions."
        )

    if "Set-Piece Threat" in high_areas:
        preparation_focus.append(
            "Set-piece defending should be a specific preparation block."
        )

    if not preparation_focus:
        preparation_focus.append(
            "No single extreme profile stands out. Prepare for a balanced opponent and focus on match-specific video."
        )

    return {
        "overview": profile["overview"],
        "style_interpretation": interpretation_df,
        "attacking_profile": attacking_profile,
        "defensive_profile": defensive_profile,
        "main_strengths": pd.DataFrame({"point": main_strengths}),
        "possible_vulnerabilities": pd.DataFrame({"point": possible_vulnerabilities}),
        "preparation_focus": pd.DataFrame({"point": preparation_focus}),
        "key_metrics": profile["key_metrics"],
        "league_rankings": profile["league_rankings"],
    }

# =========================
# BUILD OPPOSITION REPORT HELPER
# =========================

opposition = build_opposition_report_helper(
    team_style_df,
    team_name="Ajax",
    league_name="Eredivisie",
    season="2025-26",
)

opposition["overview"]

In [ ]:
opposition["style_interpretation"]
# opposition["attacking_profile"]
# opposition["defensive_profile"]
# opposition["main_strengths"]
# opposition["possible_vulnerabilities"]
# opposition["preparation_focus"]

In [ ]:
# =========================
# TEAM STYLE RADAR CONFIG
# =========================

from matplotlib import pyplot as plt
import numpy as np


TEAM_STYLE_RADAR_COLUMNS = [
    "possession_control_score",
    "territory_penetration_score",
    "attacking_volume_score",
    "pressing_ball_winning_score",
    "defensive_resistance_score",
    "directness_score",
    "width_crossing_score",
    "physicality_duels_score",
    "set_piece_threat_score",
]


TEAM_STYLE_RADAR_LABELS = {
    "possession_control_score": "Possession\nControl",
    "territory_penetration_score": "Territory &\nPenetration",
    "attacking_volume_score": "Attacking\nVolume",
    "pressing_ball_winning_score": "Pressing &\nBall Winning",
    "defensive_resistance_score": "Defensive\nResistance",
    "directness_score": "Directness",
    "width_crossing_score": "Width &\nCrossing",
    "physicality_duels_score": "Physicality /\nDuels",
    "set_piece_threat_score": "Set-Piece\nThreat",
}

# =========================
# TEAM STYLE ROW SELECTOR
# =========================

def select_team_style_row(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
) -> pd.Series:
    """
    Selects one team row from team_style_df.

    If multiple teams match, it raises an error and shows the possible candidates.
    """

    view = df.copy()

    # Filter by team name
    if exact_match:
        view = view[
            view["contestant_name"].str.lower() == team_name.lower()
        ]
    else:
        view = view[
            view["contestant_name"].str.contains(team_name, case=False, na=False)
        ]

    # Filter by league
    if league_name:
        view = view[
            view["source_league_name"].str.contains(
                league_name,
                case=False,
                na=False,
            )
        ]

    # Filter by country
    if country_name:
        view = view[
            view["source_country_name"].str.contains(
                country_name,
                case=False,
                na=False,
            )
        ]

    # Filter by season
    if season:
        view = view[view["source_season"] == season]

    if view.empty:
        raise ValueError("No team found with the selected filters.")

    if len(view) > 1:
        candidate_cols = [
            "contestant_name",
            "source_country_name",
            "source_league_name",
            "source_season",
        ]

        display(view[candidate_cols].drop_duplicates().reset_index(drop=True))

        raise ValueError(
            "Multiple teams matched. Use league_name, country_name, season, "
            "or exact_match=True to select one team."
        )

    return view.iloc[0]

# =========================
# TEAM STYLE RADAR CHART
# =========================

def plot_team_style_radar(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
    columns: list[str] = TEAM_STYLE_RADAR_COLUMNS,
    labels_map: dict[str, str] = TEAM_STYLE_RADAR_LABELS,
    title: str | None = None,
    figsize: tuple[int, int] = (8, 8),
):
    """
    Plots a radar chart for one team's style profile.

    Scores are expected to be 0-100 percentile-style values.
    """

    team_row = select_team_style_row(
        df,
        team_name=team_name,
        league_name=league_name,
        country_name=country_name,
        season=season,
        exact_match=exact_match,
    )

    # Keep only columns that exist
    available_cols = [col for col in columns if col in df.columns]

    if not available_cols:
        raise ValueError("No radar style score columns found in dataframe.")

    labels = [labels_map.get(col, col) for col in available_cols]
    values = [team_row[col] for col in available_cols]

    # Close the radar shape
    values = values + values[:1]

    angles = np.linspace(
        0,
        2 * np.pi,
        len(available_cols),
        endpoint=False,
    ).tolist()

    angles = angles + angles[:1]

    fig, ax = plt.subplots(
        figsize=figsize,
        subplot_kw={"polar": True},
    )

    # Plot radar line and fill
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.20)

    # Set axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)

    # Set score range
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20", "40", "60", "80", "100"])

    # Put first metric at the top
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    # Title
    if title is None:
        title = (
            f"{team_row['contestant_name']} Style Profile\n"
            f"{team_row['source_league_name']} | {team_row['source_season']}"
        )

    ax.set_title(title, pad=25)

    plt.tight_layout()

    return fig, ax

# =========================
# EXAMPLE: TEAM RADAR
# =========================

fig, ax = plot_team_style_radar(
    team_style_df,
    team_name="Ajax",
    league_name="Eredivisie",
    season="2025-26",
)

In [ ]:
# =========================
# SIMILAR TEAM SEARCH CONFIG
# =========================

SIMILAR_TEAM_COLUMNS = TEAM_STYLE_RADAR_COLUMNS

# =========================
# FIND SIMILAR TEAMS
# =========================

def find_similar_teams(
    df: pd.DataFrame,
    *,
    team_name: str,
    league_name: str | None = None,
    country_name: str | None = None,
    season: str | None = None,
    exact_match: bool = False,
    columns: list[str] = SIMILAR_TEAM_COLUMNS,
    top_n: int = 10,
    same_season_only: bool = True,
    same_league_only: bool = False,
    same_country_only: bool = False,
    exclude_same_team: bool = True,
) -> pd.DataFrame:
    """
    Finds teams with the most similar style profile.

    Similarity is based on the team style score columns.
    Lower distance = more similar.
    Higher similarity_score = more similar.

    Important:
    This compares relative style profiles, not team quality.
    """

    data = df.copy()

    # Select reference team
    reference_row = select_team_style_row(
        data,
        team_name=team_name,
        league_name=league_name,
        country_name=country_name,
        season=season,
        exact_match=exact_match,
    )

    available_cols = [col for col in columns if col in data.columns]

    if not available_cols:
        raise ValueError("No similarity columns found in dataframe.")

    # Candidate pool
    candidates = data.copy()

    # Compare only same season, useful when mixing 2024-25 and 2025-26
    if same_season_only:
        candidates = candidates[
            candidates["source_season"] == reference_row["source_season"]
        ]

    # Optional: compare only inside same league
    if same_league_only:
        candidates = candidates[
            candidates["source_league_slug"] == reference_row["source_league_slug"]
        ]

    # Optional: compare only inside same country
    if same_country_only:
        candidates = candidates[
            candidates["source_country_name"] == reference_row["source_country_name"]
        ]

    # Exclude reference team
    if exclude_same_team:
        candidates = candidates[
            ~(
                (candidates["contestant_name"] == reference_row["contestant_name"])
                & (candidates["source_league_slug"] == reference_row["source_league_slug"])
                & (candidates["source_season"] == reference_row["source_season"])
            )
        ]

    if candidates.empty:
        raise ValueError("No candidate teams available after filters.")

    # Reference vector
    reference_vector = reference_row[available_cols].astype(float)

    # Candidate matrix
    candidate_matrix = candidates[available_cols].astype(float)

    # Handle missing values
    candidate_matrix = candidate_matrix.fillna(candidate_matrix.mean())
    reference_vector = reference_vector.fillna(candidate_matrix.mean())

    # Euclidean distance
    distances = np.sqrt(
        ((candidate_matrix - reference_vector) ** 2).sum(axis=1)
    )

    # Convert distance to 0-100 similarity score
    max_possible_distance = np.sqrt(len(available_cols) * (100 ** 2))

    similarity_score = (
        100 * (1 - distances / max_possible_distance)
    ).clip(lower=0, upper=100)

    result = candidates.copy()
    result["style_distance"] = distances
    result["similarity_score"] = similarity_score

    output_cols = [
        "contestant_name",
        "source_country_name",
        "source_league_name",
        "source_season",
        "similarity_score",
        "style_distance",
    ] + available_cols

    return (
        result[output_cols]
        .sort_values(["similarity_score", "style_distance"], ascending=[False, True])
        .head(top_n)
        .reset_index(drop=True)
    )

similar_teams = find_similar_teams(
    team_style_df,
    team_name="Ajax",
    league_name="Eredivisie",
    season="2025-26",
    top_n=10,
)

similar_teams

In [ ]:
# =========================
# COMPARE TWO TEAM STYLE PROFILES
# =========================

def compare_team_style_profiles(
    df: pd.DataFrame,
    *,
    team_a: str,
    team_b: str,
    team_a_league: str | None = None,
    team_b_league: str | None = None,
    team_a_country: str | None = None,
    team_b_country: str | None = None,
    team_a_season: str | None = None,
    team_b_season: str | None = None,
    exact_match: bool = False,
    columns: list[str] = SIMILAR_TEAM_COLUMNS,
) -> pd.DataFrame:
    """
    Compares two teams across the style score columns.
    """

    row_a = select_team_style_row(
        df,
        team_name=team_a,
        league_name=team_a_league,
        country_name=team_a_country,
        season=team_a_season,
        exact_match=exact_match,
    )

    row_b = select_team_style_row(
        df,
        team_name=team_b,
        league_name=team_b_league,
        country_name=team_b_country,
        season=team_b_season,
        exact_match=exact_match,
    )

    available_cols = [col for col in columns if col in df.columns]

    rows = []

    for col in available_cols:
        label = TEAM_STYLE_SCORE_LABELS.get(col, col)

        value_a = row_a[col]
        value_b = row_b[col]

        rows.append(
            {
                "style_area": label,
                f"{row_a['contestant_name']}": value_a,
                f"{row_b['contestant_name']}": value_b,
                "absolute_gap": abs(value_a - value_b),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("absolute_gap", ascending=True)
        .reset_index(drop=True)
    )

comparison = compare_team_style_profiles(
    team_style_df,
    team_a="Ajax",
    team_a_league="Eredivisie",
    team_a_season="2025-26",
    team_b="PSV",
    team_b_league="Eredivisie",
    team_b_season="2025-26",
)

comparison